In [1]:

from sklearn.datasets import load_diabetes
import pandas as pd
from pathlib import Path

# Load the built-in diabetes dataset
diabetes = load_diabetes(as_frame=True)

df = diabetes.frame.copy()

df = df.rename(columns={
    "target": "DiseaseProgression"
})

# Save dataset for reproducibility
Path("data").mkdir(exist_ok=True)

df.to_csv(
    "data/diabetes_progression.csv",
    index=False
)

print("Dataset shape:", df.shape)
print("Missing values:", df.isnull().sum().sum())

display(df.head())


Dataset shape: (442, 11)
Missing values: 0


,age,sex,bmi,bp,s1,s2,s3,s4,s5,s6,DiseaseProgression
0,0.038076,0.050680,0.061696,0.021872,-0.044223,-0.034821,-0.043401,-0.002592,0.019907,-0.017646,151.0
1,-0.001882,-0.044642,-0.051474,-0.026328,-0.008449,-0.019163,0.074412,-0.039493,-0.068332,-0.092204,75.0
2,0.085299,0.050680,0.044451,-0.005670,-0.045599,-0.034194,-0.032356,-0.002592,0.002861,-0.025930,141.0
3,-0.089063,-0.044642,-0.011595,-0.036656,0.012191,0.024991,-0.036038,0.034309,0.022688,-0.009362,206.0
4,0.005383,-0.044642,-0.036385,0.021872,0.003935,0.015596,0.008142,-0.002592,-0.031988,-0.046641,135.0


In [2]:

from sklearn.model_selection import train_test_split

# Separate input features and target
X = df.drop(columns=["DiseaseProgression"])
y = df["DiseaseProgression"]

# Split data: 80% training, 20% testing
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

print("Training features:", X_train.shape)
print("Testing features:", X_test.shape)
print("Training target:", y_train.shape)
print("Testing target:", y_test.shape)

print("\nFeature columns:")
print(X.columns.tolist())

print("\nTarget summary:")
print(y.describe())


Training features: (353, 10)
Testing features: (89, 10)
Training target: (353,)
Testing target: (89,)

Feature columns:
['age', 'sex', 'bmi', 'bp', 's1', 's2', 's3', 's4', 's5', 's6']

Target summary:
count    442.000000
mean     152.133484
std       77.093005
min       25.000000
25%       87.000000
50%      140.500000
75%      211.500000
max      346.000000
Name: DiseaseProgression, dtype: float64


In [3]:

import numpy as np
import pandas as pd
import mlflow
import mlflow.sklearn

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import (
    RandomForestRegressor,
    HistGradientBoostingRegressor
)
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

# Connect to MLflow
mlflow.set_tracking_uri("http://127.0.0.1:5000")

# Separate experiment from Salary Prediction
mlflow.set_experiment("EXP13_Diabetes_Progression")

models = {
    "LinearRegression": LinearRegression(),
    "Ridge": Ridge(alpha=1.0),
    "RandomForest": RandomForestRegressor(
        n_estimators=100,
        random_state=42
    ),
    "HistGradBoost": HistGradientBoostingRegressor(
        random_state=42
    )
}

results = []

for name, estimator in models.items():

    pipeline = Pipeline([
        ("scaler", StandardScaler()),
        ("model", estimator)
    ])

    with mlflow.start_run(run_name=name) as run:

        pipeline.fit(X_train, y_train)
        predictions = pipeline.predict(X_test)

        mae = mean_absolute_error(
            y_test, predictions
        )

        rmse = np.sqrt(
            mean_squared_error(y_test, predictions)
        )

        r2 = r2_score(y_test, predictions)

        # Record evaluation metrics
        mlflow.log_metric("mae", float(mae))
        mlflow.log_metric("rmse", float(rmse))
        mlflow.log_metric("r2", float(r2))

        mlflow.log_param("algorithm", name)
        mlflow.log_param("test_size", 0.20)
        mlflow.log_param("random_state", 42)

        # Save trained pipeline in MLflow
        mlflow.sklearn.log_model(
            sk_model=pipeline,
            name="model",
            input_example=X_train.head(3),
            serialization_format="cloudpickle"
        )

        results.append({
            "Model": name,
            "MAE": mae,
            "RMSE": rmse,
            "R2": r2,
            "Run_ID": run.info.run_id
        })

results_df = pd.DataFrame(results)

results_df = results_df.sort_values(
    by="RMSE"
).reset_index(drop=True)

print("MODEL COMPARISON RESULTS")
display(results_df)

print("\nBest Model:", results_df.iloc[0]["Model"])


C:\Users\gopik\FDP_MLOPS\salary_m1\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026/11/01 18:01:27 INFO mlflow.tracking.fluent: Experiment with name 'EXP13_Diabetes_Progression' does not exist. Creating a new experiment.
2026/11/01 18:01:28 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run LinearRegression at: http://127.0.0.1:5000/#/experiments/2/runs/047e5cea4968457a84dbd93d788f9087
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2


2026/11/01 18:01:50 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run Ridge at: http://127.0.0.1:5000/#/experiments/2/runs/fe448fc8b6964b4f9da3d45c5e14b7a1
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2


2026/11/01 18:01:58 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run RandomForest at: http://127.0.0.1:5000/#/experiments/2/runs/ce061877351145119d0ad44e20d9d2a8
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2


2026/11/01 18:02:08 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run HistGradBoost at: http://127.0.0.1:5000/#/experiments/2/runs/cc8588af690640bfaec8ed3854baa746
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2
MODEL COMPARISON RESULTS


,Model,MAE,RMSE,R2,Run_ID
0,Ridge,42.811999,53.777454,0.454147,fe448fc8b6964b4f9da3d45c5e14b7a1
1,LinearRegression,42.794095,53.853446,0.452603,047e5cea4968457a84dbd93d788f9087
2,RandomForest,44.171236,54.398351,0.441469,ce061877351145119d0ad44e20d9d2a8
3,HistGradBoost,45.022682,57.008045,0.386594,cc8588af690640bfaec8ed3854baa746



Best Model: Ridge


In [4]:

import mlflow
import mlflow.sklearn
from mlflow.tracking import MlflowClient

mlflow.set_tracking_uri("http://127.0.0.1:5000")

MODEL_NAME = "DiabetesProgressionModel"

# Automatically identify the best run
best_row = results_df.iloc[0]
best_run_id = best_row["Run_ID"]

experiment = mlflow.get_experiment_by_name(
    "EXP13_Diabetes_Progression"
)

# Find the model already logged during training
logged_models = mlflow.search_logged_models(
    experiment_ids=[experiment.experiment_id],
    filter_string=f"source_run_id = '{best_run_id}'",
    output_format="list"
)

if not logged_models:
    raise RuntimeError("Best run has no logged model.")

model_id = logged_models[0].model_id

# Register the existing model
registered = mlflow.register_model(
    model_uri=f"models:/{model_id}",
    name=MODEL_NAME
)

# Assign champion alias
client = MlflowClient()

client.set_registered_model_alias(
    name=MODEL_NAME,
    alias="champion",
    version=registered.version
)

print("Registered Model:", MODEL_NAME)
print("Algorithm:", best_row["Model"])
print("Registered Version:", registered.version)
print("Champion Alias: champion")
print("RMSE:", round(best_row["RMSE"], 4))


Successfully registered model 'DiabetesProgressionModel'.
2026/11/01 18:04:47 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: DiabetesProgressionModel, version 1
Created version '1' of model 'DiabetesProgressionModel'.


Registered Model: DiabetesProgressionModel
Algorithm: Ridge
Registered Version: 1
Champion Alias: champion
RMSE: 53.7775
